# ETL-19 · dim_tempo e dim_gravidade

- **Etapa:** ETL-19, dim_tempo e dim_gravidade
- **Responsável:** Davi Mello (tarefa assumida do Samuel)
- **Issue:** ETL-19
- **Entradas:** `data/intermediarios/04_datas.parquet` (datas convertidas, ETL-15) e `data/intermediarios/05_nulos_categorias.parquet` (`gravidade_lesao`, ETL-16)
- **Saídas:** `data/intermediarios/dims/dim_tempo.parquet` e `dim_gravidade.parquet`

Cria o calendário do modelo estrela (usado duas vezes pela fato: data do sinistro e data do óbito) e a dimensão de gravidade da lesão. As duas têm os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). As colunas seguem o DDL da ETL-18 (`sql/ddl/etl_star_schema.sql`).

## 1. Setup

In [1]:
from pathlib import Path
import pandas as pd

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

INTERM = RAIZ / "data" / "intermediarios"      # .parquet (o .gitignore já ignora)
DIMS   = INTERM / "dims"                       # saída: uma pasta só para as dimensões
DIMS.mkdir(parents=True, exist_ok=True)

# Na execução final as duas variáveis apontam para o mesmo arquivo (a saída da etapa anterior)
ENTRADA_DATAS     = INTERM / "04_datas.parquet"             # saída da ETL-15: datas já convertidas
ENTRADA_GRAVIDADE = INTERM / "05_nulos_categorias.parquet"  # saída da ETL-16: gravidade_lesao ainda com NAO DISPONIVEL

INICIO = pd.Timestamp("2021-01-01")   # primeiro dia do período do projeto

## 2. Carregar a entrada

In [2]:
df_datas = pd.read_parquet(ENTRADA_DATAS, columns=['id_pessoa', 'data_sinistro', 'data_obito'])
df_grav  = pd.read_parquet(ENTRADA_GRAVIDADE, columns=['id_pessoa', 'gravidade_lesao', 'flag_obito'])
print('datas:', df_datas.shape, '| gravidade:', df_grav.shape)

# os dois arquivos têm que ser da mesma base: mesmas pessoas
assert len(df_datas) == len(df_grav) and df_datas['id_pessoa'].is_unique
assert df_datas['id_pessoa'].isin(df_grav['id_pessoa']).all()
print('mesmas', len(df_datas), 'pessoas nos dois arquivos')

datas: (806977, 3) | gravidade: (806977, 3)
mesmas 806977 pessoas nos dois arquivos


In [3]:
def para_data(serie: pd.Series) -> pd.Series:
    """Aceita a data já convertida (ETL-15) ou o texto dd/mm/aaaa da fonte, e devolve só a data."""
    if pd.api.types.is_datetime64_any_dtype(serie):
        return serie.dt.normalize()
    return pd.to_datetime(serie, format='%d/%m/%Y', errors='raise')

df_datas['data_sinistro'] = para_data(df_datas['data_sinistro'])
df_datas['data_obito'] = para_data(df_datas['data_obito'])

## 3. Diagnóstico ANTES

In [4]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela-resumo usada antes e depois: tipo, % de nulos e nº de valores distintos por coluna."""
    return pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'pct_nulo': (df.isna().mean() * 100).round(2),
        'n_distintos': df.nunique(),
    })

antes = pd.concat([
    diagnostico(df_datas[['data_sinistro', 'data_obito']]),
    diagnostico(df_grav[['gravidade_lesao']]),
])
antes

,dtype,pct_nulo,n_distintos
data_sinistro,datetime64[us],0.00,1095
data_obito,datetime64[us],98.04,1109
gravidade_lesao,str,0.00,5


In [5]:
# período real das datas da base
print('data_sinistro:', df_datas['data_sinistro'].min().date(), 'a', df_datas['data_sinistro'].max().date())
print('data_obito:   ', df_datas['data_obito'].min().date(), 'a', df_datas['data_obito'].max().date())
print('\ngravidade_lesao:')
print(df_grav['gravidade_lesao'].value_counts(dropna=False))

data_sinistro: 2021-01-01 a 2023-12-31
data_obito:    2021-01-01 a 2024-01-24

gravidade_lesao:
gravidade_lesao
LEVE              357297
ILESO             321999
NAO DISPONIVEL     70314
GRAVE              41574
FATAL              15793
Name: count, dtype: int64


## 4. dim_tempo

Um dia por linha, de 01/01/2021 até a maior data da base (a maior `data_obito`, porque há óbitos até 30 dias depois de sinistros de dezembro de 2023). A chave `sk_tempo` é a própria data no formato `AAAAMMDD` (ex.: `20210115`), como define o dicionário do DW.

### 4.1 Feriados nacionais

Só feriados nacionais, como proposto no modelo estrela (ETL-10). São os 8 de data fixa mais a Sexta-feira Santa, que depende da Páscoa. Carnaval e Corpus Christi ficam de fora porque são ponto facultativo, não feriado. O Dia da Consciência Negra (20/11) só virou feriado nacional em 2024 (Lei 14.759/2023), depois do período da base, mas a regra já está na função.

In [6]:
from datetime import timedelta

def pascoa(ano: int) -> pd.Timestamp:
    """Domingo de Páscoa do ano (algoritmo de Meeus/Jones/Butcher, calendário gregoriano)."""
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = (h + l - 7 * m + 114) % 31 + 1
    return pd.Timestamp(ano, mes, dia)

# conferência com as datas conhecidas da Páscoa
assert [pascoa(a).date().isoformat() for a in (2021, 2022, 2023, 2024)] == \
    ['2021-04-04', '2022-04-17', '2023-04-09', '2024-03-31']

FERIADOS_FIXOS = [
    (1, 1, 'Confraternização Universal'),
    (4, 21, 'Tiradentes'),
    (5, 1, 'Dia do Trabalho'),
    (9, 7, 'Independência do Brasil'),
    (10, 12, 'Nossa Senhora Aparecida'),
    (11, 2, 'Finados'),
    (11, 15, 'Proclamação da República'),
    (12, 25, 'Natal'),
]

def feriados_nacionais(anos) -> pd.Series:
    """Série data -> nome do feriado nacional, para os anos pedidos."""
    feriados = {}
    for ano in anos:
        for mes, dia, nome in FERIADOS_FIXOS:
            feriados[pd.Timestamp(ano, mes, dia)] = nome
        feriados[pd.Timestamp(pascoa(ano).to_pydatetime() - timedelta(days=2))] = 'Sexta-feira Santa'
        if ano >= 2024:   # Lei 14.759/2023
            feriados[pd.Timestamp(ano, 11, 20)] = 'Dia Nacional de Zumbi e da Consciência Negra'
    return pd.Series(feriados, name='nome_feriado').sort_index()

### 4.2 Criar a dimensão

In [7]:
NOMES_MES = ['JANEIRO', 'FEVEREIRO', 'MARÇO', 'ABRIL', 'MAIO', 'JUNHO', 'JULHO',
             'AGOSTO', 'SETEMBRO', 'OUTUBRO', 'NOVEMBRO', 'DEZEMBRO']
NOMES_DIA = ['SEGUNDA-FEIRA', 'TERÇA-FEIRA', 'QUARTA-FEIRA', 'QUINTA-FEIRA',
             'SEXTA-FEIRA', 'SÁBADO', 'DOMINGO']   # 1 = segunda ... 7 = domingo

def criar_dim_tempo(inicio: pd.Timestamp, fim: pd.Timestamp) -> pd.DataFrame:
    """Um dia por linha entre inicio e fim, com chave AAAAMMDD e os membros -1 e -2."""
    datas = pd.date_range(inicio, fim, freq='D')
    feriados = feriados_nacionais(range(inicio.year, fim.year + 1))

    dim = pd.DataFrame({'data': datas})
    dim.insert(0, 'sk_tempo', (datas.year * 10000 + datas.month * 100 + datas.day).astype('int64'))
    dim['ano'] = datas.year
    dim['trimestre'] = datas.quarter
    dim['mes'] = datas.month
    dim['nome_mes'] = [NOMES_MES[m - 1] for m in datas.month]
    dim['dia'] = datas.day
    dim['ano_mes'] = datas.strftime('%Y-%m')
    dim['dia_semana'] = datas.day_of_week + 1
    dim['nome_dia_semana'] = [NOMES_DIA[d] for d in datas.day_of_week]
    dim['fim_de_semana'] = dim['dia_semana'].isin([6, 7])
    dim['nome_feriado'] = dim['data'].map(feriados)
    dim['feriado'] = dim['nome_feriado'].notna()

    # mesma ordem de colunas do DDL
    dim = dim[['sk_tempo', 'data', 'ano', 'trimestre', 'mes', 'nome_mes', 'dia', 'ano_mes',
               'dia_semana', 'nome_dia_semana', 'fim_de_semana', 'feriado', 'nome_feriado']]

    # tipos que aceitam nulo, compatíveis com o DDL (SMALLINT, BOOLEAN, VARCHAR)
    for col in ['ano', 'trimestre', 'mes', 'dia', 'dia_semana']:
        dim[col] = dim[col].astype('Int16')
    for col in ['fim_de_semana', 'feriado']:
        dim[col] = dim[col].astype('boolean')
    TEXTOS = ['nome_mes', 'ano_mes', 'nome_dia_semana', 'nome_feriado']
    for col in TEXTOS:
        dim[col] = dim[col].astype('string')

    # membros especiais, com os mesmos tipos: números e datas ficam nulos, textos recebem o rótulo
    especiais = pd.DataFrame({c: pd.Series([pd.NA, pd.NA], dtype=dim[c].dtype) for c in dim.columns[1:]})
    especiais.insert(0, 'sk_tempo', pd.Series([-1, -2], dtype='int64'))
    for col in ['nome_mes', 'ano_mes', 'nome_dia_semana']:
        especiais[col] = pd.Series(['NÃO INFORMADO', 'NÃO SE APLICA'], dtype='string')
    dim = pd.concat([especiais, dim], ignore_index=True)
    return dim

# o calendário vai até a maior data que aparece na base
FIM = max(df_datas['data_sinistro'].max(), df_datas['data_obito'].max())
assert df_datas['data_sinistro'].min() >= INICIO, 'há sinistro antes de 2021'

dim_tempo = criar_dim_tempo(INICIO, FIM)
print(len(dim_tempo), 'linhas | de', INICIO.date(), 'a', FIM.date())
dim_tempo.head(6)

1121 linhas | de 2021-01-01 a 2024-01-24


,sk_tempo,data,ano,trimestre,mes,nome_mes,dia,ano_mes,dia_semana,nome_dia_semana,fim_de_semana,feriado,nome_feriado
0,-1,NaT,<NA>,<NA>,<NA>,NÃO INFORMADO,<NA>,NÃO INFORMADO,<NA>,NÃO INFORMADO,<NA>,<NA>,<NA>
1,-2,NaT,<NA>,<NA>,<NA>,NÃO SE APLICA,<NA>,NÃO SE APLICA,<NA>,NÃO SE APLICA,<NA>,<NA>,<NA>
2,20210101,2021-01-01,2021,1,1,JANEIRO,1,2021-01,5,SEXTA-FEIRA,False,True,Confraternização Universal
3,20210102,2021-01-02,2021,1,1,JANEIRO,2,2021-01,6,SÁBADO,True,False,<NA>
4,20210103,2021-01-03,2021,1,1,JANEIRO,3,2021-01,7,DOMINGO,True,False,<NA>
5,20210104,2021-01-04,2021,1,1,JANEIRO,4,2021-01,1,SEGUNDA-FEIRA,False,False,<NA>


In [8]:
# feriados de cada ano (dias marcados na dimensão)
reais = dim_tempo[dim_tempo['sk_tempo'] > 0]
feriados_por_ano = reais[reais['feriado']].groupby('ano')['nome_feriado'].count()
print(feriados_por_ano.to_string(), '\n')
reais.loc[reais['feriado'], ['data', 'nome_dia_semana', 'nome_feriado']].head(10)

ano
2021    9
2022    9
2023    9
2024    1 



,data,nome_dia_semana,nome_feriado
2,2021-01-01,SEXTA-FEIRA,Confraternização Universal
93,2021-04-02,SEXTA-FEIRA,Sexta-feira Santa
112,2021-04-21,QUARTA-FEIRA,Tiradentes
122,2021-05-01,SÁBADO,Dia do Trabalho
251,2021-09-07,TERÇA-FEIRA,Independência do Brasil
286,2021-10-12,TERÇA-FEIRA,Nossa Senhora Aparecida
307,2021-11-02,TERÇA-FEIRA,Finados
320,2021-11-15,SEGUNDA-FEIRA,Proclamação da República
360,2021-12-25,SÁBADO,Natal
367,2022-01-01,SÁBADO,Confraternização Universal


## 5. dim_gravidade

As quatro gravidades da fonte, com `nivel_gravidade` de 0 (ileso) a 3 (fatal) para ordenar e filtrar ("grave ou pior" = nível 2 ou mais). A ETL-16 deixou `NAO DISPONIVEL` em `gravidade_lesao` para esta etapa: ele vai para o membro -1.

In [9]:
NIVEIS = {'ILESO': 0, 'LEVE': 1, 'GRAVE': 2, 'FATAL': 3}
NAO_INFORMADO = ['NAO DISPONIVEL', 'NÃO INFORMADO']

def criar_dim_gravidade(gravidade: pd.Series) -> pd.DataFrame:
    """Uma linha por gravidade, chave na ordem do nível (1 = ILESO ... 4 = FATAL), e os membros -1 e -2."""
    valores = set(gravidade.dropna().unique()) - set(NAO_INFORMADO)
    # só aceita as gravidades conhecidas, para nunca criar uma linha de lixo na dimensão
    assert valores <= set(NIVEIS), f'gravidade nova: {valores - set(NIVEIS)}'

    dim = pd.DataFrame({'gravidade_lesao': list(NIVEIS), 'nivel_gravidade': list(NIVEIS.values())})
    dim.insert(0, 'sk_gravidade', dim['nivel_gravidade'] + 1)
    especiais = pd.DataFrame({
        'sk_gravidade': [-1, -2],
        'gravidade_lesao': ['NÃO INFORMADO', 'NÃO SE APLICA'],
        'nivel_gravidade': [pd.NA, pd.NA],
    })
    dim = pd.concat([especiais, dim], ignore_index=True)
    dim['sk_gravidade'] = dim['sk_gravidade'].astype('int64')
    dim['gravidade_lesao'] = dim['gravidade_lesao'].astype('string')
    dim['nivel_gravidade'] = dim['nivel_gravidade'].astype('Int16')
    return dim

dim_gravidade = criar_dim_gravidade(df_grav['gravidade_lesao'])
dim_gravidade

,sk_gravidade,gravidade_lesao,nivel_gravidade
0,-1,NÃO INFORMADO,<NA>
1,-2,NÃO SE APLICA,<NA>
2,1,ILESO,0
3,2,LEVE,1
4,3,GRAVE,2
5,4,FATAL,3


## 6. Conferir e salvar

As funções `atribuir_sk_tempo` e `atribuir_sk_gravidade` ligam cada pessoa à chave, e a ETL-23 (tabela fato) pode reutilizá-las. Conferências: chaves únicas, calendário sem buracos, tamanhos do DDL respeitados e todas as pessoas com chave.

In [10]:
def atribuir_sk_tempo(datas: pd.Series, sem_data: int) -> pd.Series:
    """Chave AAAAMMDD de cada data; quem não tem data recebe sem_data (-2 no óbito de quem não morreu)."""
    datas = para_data(datas)
    sk = datas.dt.year * 10000 + datas.dt.month * 100 + datas.dt.day
    return sk.fillna(sem_data).astype('int64')

def atribuir_sk_gravidade(gravidade: pd.Series) -> pd.Series:
    """Chave de cada pessoa na dim_gravidade (-1 quando NAO DISPONIVEL ou vazia)."""
    sk = gravidade.map({g: n + 1 for g, n in NIVEIS.items()})
    return sk.fillna(-1).astype('int64')

In [11]:
# 1. chaves únicas e calendário contínuo, sem buracos
reais = dim_tempo[dim_tempo['sk_tempo'] > 0]
assert dim_tempo['sk_tempo'].is_unique and reais['data'].is_unique
assert reais['sk_tempo'].is_monotonic_increasing
assert len(reais) == (FIM - INICIO).days + 1, 'faltam dias no calendário'
assert dim_gravidade['sk_gravidade'].is_unique and dim_gravidade['gravidade_lesao'].is_unique

# 2. tamanhos dos textos dentro do DDL
LIMITES = [
    (dim_tempo, {'nome_mes': 20, 'ano_mes': 15, 'nome_dia_semana': 20, 'nome_feriado': 40}),
    (dim_gravidade, {'gravidade_lesao': 20}),
]
for dim, limites in LIMITES:
    for col, limite in limites.items():
        assert dim[col].str.len().max() <= limite, f'{col} maior que {limite}'

# 3. dias reais com todos os campos preenchidos (CHECK chk_dim_tempo_campos_validos)
obrigatorias = ['data', 'ano', 'trimestre', 'mes', 'nome_mes', 'dia', 'ano_mes',
                'dia_semana', 'nome_dia_semana', 'fim_de_semana', 'feriado']
assert reais[obrigatorias].notna().all().all()

for nome, dim in [('dim_tempo', dim_tempo), ('dim_gravidade', dim_gravidade)]:
    dim.to_parquet(DIMS / f'{nome}.parquet', index=False)
    print(nome, '|', len(dim), 'linhas | salvo em dims/')

dim_tempo | 1121 linhas | salvo em dims/
dim_gravidade | 6 linhas | salvo em dims/


In [12]:
# 4. cada pessoa tem que achar uma chave
sk_data_sinistro = atribuir_sk_tempo(df_datas['data_sinistro'], sem_data=-1)
sk_data_obito    = atribuir_sk_tempo(df_datas['data_obito'], sem_data=-2)
sk_gravidade     = atribuir_sk_gravidade(df_grav['gravidade_lesao'])

for nome, sk, dim, col in [('data do sinistro', sk_data_sinistro, dim_tempo, 'sk_tempo'),
                           ('data do óbito', sk_data_obito, dim_tempo, 'sk_tempo'),
                           ('gravidade', sk_gravidade, dim_gravidade, 'sk_gravidade')]:
    assert sk.isin(dim[col]).all(), f'{nome}: chave fora da dimensão'
    print(f'{nome:16s} | pessoas com chave: {len(sk)} | -1: {int((sk == -1).sum())} | -2: {int((sk == -2).sum())}')

# o -2 da data do óbito tem que ser exatamente quem não morreu
assert ((sk_data_obito == -2) == ~df_grav['flag_obito'].to_numpy()).all()

data do sinistro | pessoas com chave: 806977 | -1: 0 | -2: 0
data do óbito    | pessoas com chave: 806977 | -1: 0 | -2: 791184
gravidade        | pessoas com chave: 806977 | -1: 70314 | -2: 0


In [13]:
# Diagnóstico DEPOIS: tamanho das dimensões e uso dos membros especiais
depois = pd.DataFrame({
    'linhas': [len(dim_tempo), len(dim_gravidade)],
    'linhas_reais': [int((dim_tempo['sk_tempo'] > 0).sum()), int((dim_gravidade['sk_gravidade'] > 0).sum())],
    'pessoas_-1 (NÃO INFORMADO)': [int((sk_data_sinistro == -1).sum() + (sk_data_obito == -1).sum()),
                                   int((sk_gravidade == -1).sum())],
    'pessoas_-2 (NÃO SE APLICA)': [int((sk_data_obito == -2).sum()), int((sk_gravidade == -2).sum())],
}, index=['dim_tempo', 'dim_gravidade'])
depois

,linhas,linhas_reais,pessoas_-1 (NÃO INFORMADO),pessoas_-2 (NÃO SE APLICA)
dim_tempo,1121,1119,0,791184
dim_gravidade,6,4,70314,0


In [14]:
# gravidade: pessoas por linha da dimensão
pessoas = sk_gravidade.value_counts()
dim_gravidade.assign(pessoas=dim_gravidade['sk_gravidade'].map(pessoas).fillna(0).astype('int64'))

,sk_gravidade,gravidade_lesao,nivel_gravidade,pessoas
0,-1,NÃO INFORMADO,<NA>,70314
1,-2,NÃO SE APLICA,<NA>,0
2,1,ILESO,0,321999
3,2,LEVE,1,357297
4,3,GRAVE,2,41574
5,4,FATAL,3,15793


In [15]:
# só consulta: pessoas envolvidas por tipo de dia (usa a data do sinistro), em 2021–2023
def tipo_de_dia(fim_de_semana: pd.Series, feriado: pd.Series) -> pd.Series:
    """FERIADO vale mais que FIM DE SEMANA; o resto é DIA ÚTIL."""
    tipo = pd.Series('DIA ÚTIL', index=fim_de_semana.index)
    tipo[fim_de_semana.to_numpy(dtype=bool)] = 'FIM DE SEMANA'
    tipo[feriado.to_numpy(dtype=bool)] = 'FERIADO'
    return tipo

ORDEM = ['DIA ÚTIL', 'FIM DE SEMANA', 'FERIADO']
cal = reais[reais['ano'] <= 2023]
dias_cal = tipo_de_dia(cal['fim_de_semana'], cal['feriado']).value_counts().reindex(ORDEM)

do_sinistro = dim_tempo.set_index('sk_tempo').loc[sk_data_sinistro, ['fim_de_semana', 'feriado']].reset_index(drop=True)
pessoas_dia = tipo_de_dia(do_sinistro['fim_de_semana'], do_sinistro['feriado']).value_counts().reindex(ORDEM)

pd.DataFrame({'dias no calendário': dias_cal, 'pessoas': pessoas_dia,
              'pessoas por dia': (pessoas_dia / dias_cal).round(0).astype('int64')})

,dias no calendário,pessoas,pessoas por dia
DIA ÚTIL,760,554750,730
FIM DE SEMANA,308,234889,763
FERIADO,27,17338,642


## 7. Resumo

**O que foi feito:** criação da `dim_tempo` e da `dim_gravidade`, com as colunas do DDL da ETL-18 e os membros especiais -1 (NÃO INFORMADO) e -2 (NÃO SE APLICA). Os arquivos foram salvos em `data/intermediarios/dims/`.

**Resultado:**
- `dim_tempo`: 1.121 linhas, sendo 1.119 dias (01/01/2021 a 24/01/2024, a maior data de óbito) e os 2 membros especiais. A chave é a data no formato `AAAAMMDD`. Tem trimestre, nome do mês, `ano_mes`, dia da semana (1 = segunda ... 7 = domingo), fim de semana e 28 feriados nacionais (9 por ano em 2021–2023 e 1 em janeiro de 2024).
- `dim_gravidade`: 6 linhas, sendo ILESO, LEVE, GRAVE e FATAL (com `nivel_gravidade` de 0 a 3) e os 2 membros especiais.

**Feriados:** só nacionais, como proposto no modelo estrela: os 8 de data fixa mais a Sexta-feira Santa, calculada a partir da Páscoa. Carnaval e Corpus Christi são ponto facultativo e ficam de fora. Os feriados estaduais de SP (como 9 de julho) também ficam de fora; se a equipe quiser, basta acrescentar na lista.

**Membros especiais:** o `NAO DISPONIVEL` de `gravidade_lesao` vai para o -1 (70.314 pessoas, 8,71%). Na data do óbito, o -2 é exatamente quem não morreu (791.184 pessoas). Na `dim_tempo`, os membros especiais têm data e números nulos, e os textos recebem o rótulo, como permite o DDL. O -2 da `dim_gravidade` não é usado; ele existe para seguir a regra de que toda dimensão tem os dois membros.

**O que foi conferido:** o calendário não tem buracos, as chaves são únicas, os textos cabem nos tamanhos do DDL, os dias reais têm todos os campos obrigatórios e as 806.977 pessoas acham chave nas duas datas e na gravidade. A gravidade FATAL coincide com os 15.793 óbitos.

**Para a ETL-23 (tabela fato):** as funções `atribuir_sk_tempo` e `atribuir_sk_gravidade` mostram como ligar cada pessoa às chaves.